# Stage 6 — Test Metric Extraction

Run the per-test metric extractors against cached pipeline outputs.

**What this notebook covers:**
- Running all five extractors: explosiveness, speed, fitness, agility, balance
- Viewing `TestResult` outputs for each student track
- Comparing against ground truth (if available)
- Identifying low-confidence results and flag reasons

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import numpy as np

JOB_ID    = 'notebook-eval-detection'
TEST_TYPE = 'explosiveness'   # change to: speed, fitness, agility, balance

from pipeline.cache import PipelineCache
cache = PipelineCache(job_id=JOB_ID, cache_root=Path('../data/cache'))
print("Stages cached:", cache.summary().get('stages_cached', []))

In [ ]:
# ── Load all cached pipeline outputs ────────────────────────────────────────
from pipeline.ingest import extract_frames
from pathlib import Path

VIDEO_PATH = Path('../data/raw_footage/sample_clip.mp4')
TARGET_FPS = 15

frames = [f for _, f, _ in extract_frames(str(VIDEO_PATH), target_fps=TARGET_FPS)]

all_tracks = cache.load_tracks() if cache.has('track') else [[] for _ in frames]
all_poses  = cache.load_poses()  if cache.has('pose')  else [[] for _ in frames]
calibration = cache.load_calibration() if cache.has('calibrate') else None

if calibration is None:
    from pipeline.calibrate import Calibrator
    calibrator = Calibrator()
    calibration = calibrator.calibrate_single_axis(frames[0]) if frames else None

print(f"Frames : {len(frames)}")
print(f"Tracks : {sum(len(t) for t in all_tracks)} total")
print(f"Poses  : {sum(len(p) for p in all_poses)} total")
print(f"Calib  : {calibration.method if calibration else 'None'}  valid={calibration.is_valid if calibration else False}")

In [ ]:
import json
from pathlib import Path

configs_dir = Path('../configs/test_configs')
config_file = configs_dir / f"{TEST_TYPE}.json"
geometry_config = {}
if config_file.exists():
    with open(config_file) as f:
        geometry_config = json.load(f)
    print(f"Loaded config for '{TEST_TYPE}':")
    print(json.dumps(geometry_config, indent=2))
else:
    print(f"No config file found for '{TEST_TYPE}' — using defaults.")

In [ ]:
from worker.celery_app import _get_extractor

extractor = _get_extractor(TEST_TYPE, geometry_config, calibration)
results   = extractor.extract(all_tracks, all_poses, frames)

cache.save_results(results)
print(f"\n=== {TEST_TYPE.upper()} RESULTS ({len(results)} records) ===")
for r in results:
    flags = f"  flags={r.flags}" if r.flags else ""
    print(f"  Bib {r.student_bib:>3d} (track {r.track_id})  "
          f"{r.metric_value:.2f} {r.metric_unit}  "
          f"conf={r.confidence_score:.2f}{flags}")

In [ ]:
# ── Visualise result comparison (if ground truth available) ──────────────────
import matplotlib.pyplot as plt

# Provide ground truth as {bib_number: metric_value}
GROUND_TRUTH = {}   # e.g. {7: 34.2, 14: 28.8}

bibs    = [r.student_bib for r in results]
values  = [r.metric_value for r in results]
confs   = [r.confidence_score for r in results]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(results)), values, color=[
    'steelblue' if not r.flags else 'tomato' for r in results])
axes[0].set_xticks(range(len(results)))
axes[0].set_xticklabels([f"Bib {b}" for b in bibs], rotation=45, ha='right')
axes[0].set_ylabel(results[0].metric_unit if results else '')
axes[0].set_title(f'{TEST_TYPE} — Metric Values')

axes[1].bar(range(len(results)), confs, color='steelblue')
axes[1].axhline(0.6, color='red', linestyle='--', label='threshold=0.6')
axes[1].set_xticks(range(len(results)))
axes[1].set_xticklabels([f"Bib {b}" for b in bibs], rotation=45, ha='right')
axes[1].set_ylabel('Confidence'); axes[1].set_title('Confidence scores'); axes[1].legend()

plt.tight_layout(); plt.show()

if GROUND_TRUTH:
    print("\nAccuracy vs ground truth:")
    for r in results:
        gt = GROUND_TRUTH.get(r.student_bib)
        if gt is not None:
            error = abs(r.metric_value - gt)
            print(f"  Bib {r.student_bib:>3d}: predicted={r.metric_value:.2f}  gt={gt:.2f}  error={error:.2f} {r.metric_unit}")